Import Libraries

In [ ]:
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from transformers import pipeline

print("✅ All libraries loaded successfully")


✅ All libraries loaded successfully


Google Drive

In [ ]:
drive.mount('./documnets')

print("✅ Connected")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive connected


PDF Load

In [ ]:
DATA_PATH = "/content/Generative-AI-RAG-based-Enterprise-Knowledge-Assistant/documnets"

documents = []

for file in os.listdir(DATA_PATH):

    if file.lower().endswith(".pdf"):

        path = os.path.join(DATA_PATH, file)

        print("Loading:", file)

        loader = PyPDFLoader(path)

        documents.extend(loader.load())


print("Total pages:", len(documents))


Loading: Health & Wellness.pdf
Total pages: 8


Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))


Total chunks: 22


Embeddings

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model ready")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model ready


FAISS

In [ ]:
vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)

print("✅ FAISS vector database created")


✅ FAISS vector database created


Retriever

In [ ]:
retriever = vector_db.as_retriever(
    search_kwargs={"k":8}
)

print("✅ Retriever ready")


✅ Retriever ready


Free LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("✅ FLAN-T5 model loaded")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ FLAN-T5 model loaded


RAG Evaluation

In [ ]:
def evaluate_response(question, answer, docs):

    print("\n📊 RAG Evaluation")
    print("-"*40)

    context = " ".join([doc.page_content for doc in docs]).lower()
    answer = answer.lower()

    # Retrieval
    if docs:
        print("✅ Retrieval: Documents Found")
    else:
        print("❌ Retrieval: No Documents Found")

    # Groundedness
    overlap = sum(
        1 for word in answer.split()
        if len(word) > 4 and word in context
    )

    if overlap > 15:
        print("✅ Groundedness: High")
    elif overlap > 5:
        print("⚠️ Groundedness: Medium")
    else:
        print("❌ Groundedness: Low")

    # Completeness
    if len(answer.split()) > 20:
        print("✅ Answer Completeness: Good")
    else:
        print("⚠️ Answer Completeness: Short")

    print(f"📄 Retrieved Chunks: {len(docs)}")

Ask Question Function

In [ ]:
def ask_question(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
You are an insurance policy assistant.

Answer only from the provided context.

If multiple sections contain relevant information, combine them.

Do not omit important requirements like documents, timelines, eligibility,
or exceptions.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )


    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )


    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


    print("\nAnswer:")
    print(answer)


    print("\nSources:")

    for doc in docs:
        print(doc.metadata)

    evaluate_response(question, answer, docs)

In [ ]:
ask_question(
    "when employees can add dependents?"
)



Answer:
in an event of marriage and childbirth

Sources:
{'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Health & Wellness', 'source': '/content/drive/MyDrive/RAG_Documents/Health & Wellness.pdf', 'total_pages': 8, 'page': 1, 'page_label': '2'}
{'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Health & Wellness', 'source': '/content/drive/MyDrive/RAG_Documents/Health & Wellness.pdf', 'total_pages': 8, 'page': 2, 'page_label': '3'}
{'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Health & Wellness', 'source': '/content/drive/MyDrive/RAG_Documents/Health & Wellness.pdf', 'total_pages': 8, 'page': 2, 'page_label': '3'}
{'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Health & Wellness', 'source': '/content/drive/MyDrive/RAG_Documents/Health & Wellness.pdf', 'total_pages': 8, 'page': 

In [ ]:
import time
import sys

def chat():

    print("🤖 Insurance Policy Assistant")
    print("Type 'exit' anytime to stop\n")

    while True:

        question = input("You: ")

        if question.lower() == "exit":
            print("Assistant: Goodbye!")
            break

        print("\n⏳ Processing your question...", end="")

        # small animation
        for i in range(3):
            time.sleep(0.5)
            print(".", end="", flush=True)

        print("\n")

        ask_question(question)

        print("\n" + "-"*60 + "\n")


chat()

🤖 Insurance Policy Assistant
Type 'exit' anytime to stop


⏳ Processing your question......


Answer:
Provide relevant information.

Sources:
{'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Health & Wellness', 'source': '/content/drive/MyDrive/RAG_Documents/Health & Wellness.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}
{'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Health & Wellness', 'source': '/content/drive/MyDrive/RAG_Documents/Health & Wellness.pdf', 'total_pages': 8, 'page': 7, 'page_label': '8'}
{'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Health & Wellness', 'source': '/content/drive/MyDrive/RAG_Documents/Health & Wellness.pdf', 'total_pages': 8, 'page': 1, 'page_label': '2'}
{'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Health & Wellness', 'source': '/co